# 0. SETTING (Public Data API & SOLAR-PRO2 API)

In [1]:
from openai import OpenAI

import xml.etree.ElementTree as ET

from typing import List, Dict, Any, Tuple, Optional

import json, requests,os, time

import os
import sys
import json
import time
import uuid
import requests
import xml.etree.ElementTree as ET
import csv
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any, Optional
from openai import OpenAI

In [2]:
import os
SOLAR_API_KEY = os.getenv("SOLAR_API_KEY", "")
SERVICE_KEY = os.getenv("SERVICE_KEY", "")

In [3]:
client = OpenAI(
    api_key=SOLAR_API_KEY,
    base_url="https://api.upstage.ai/v1"
)
 
stream = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {
            "role": "user",
            "content": "HI"
        }
    ],
    stream=True,
)
 
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")

Hello! How are you today? Is there something I can help you with? 😊  

(Just let me know if you'd like to chat, ask a question, or need assistance with anything!)

In [4]:
url = 'https://apis.data.go.kr/B552657/ErmctInfoInqireService/getEmrrmRltmUsefulSckbdInfoInqire'

params = {
    'serviceKey': SERVICE_KEY,
    'STAGE1': '서울특별시',
    'STAGE2': '강남구',
    'pageNo': '1',
    'numOfRows': '10',
}

response = requests.get(url, params=params)
print(response.status_code)
print(response.text[:1000])

200
<?xml version="1.0" encoding="UTF-8" standalone="yes"?><response><header><resultCode>00</resultCode><resultMsg>NORMAL SERVICE.</resultMsg></header><body><items><item><dutyName>연세대학교의과대학강남세브란스병원</dutyName><dutyTel3>02-2019-3333</dutyTel3><hpid>A1100015</hpid><hv10>Y</hv10><hv11>Y</hv11><hv28>2</hv28><hv29>1</hv29><hv30>2</hv30><hv34>1</hv34><hv41>4</hv41><hv42>Y</hv42><hv5>Y</hv5><hv6>2</hv6><hv7>Y</hv7><hvamyn>Y</hvamyn><hvangioayn>Y</hvangioayn><hvcrrtayn>Y</hvcrrtayn><hvctayn>Y</hvctayn><hvec>16</hvec><hvecmoayn>Y</hvecmoayn><hvgc>86</hvgc><hvhypoayn>Y</hvhypoayn><hvicc>6</hvicc><hvidate>20251129071227</hvidate><hvincuayn>Y</hvincuayn><hvmriayn>Y</hvmriayn><hvncc>0</hvncc><hvoc>23</hvoc><hvoxyayn>N1</hvoxyayn><hvs01>21</hvs01><hvs02>2</hvs02><hvs03>1</hvs03><hvs04>2</hvs04><hvs08>28</hvs08><hvs12>12</hvs12><hvs15>8</hvs15><hvs17>29</hvs17><hvs22>23</hvs22><hvs25>5</hvs25><hvs26>1</hvs26><hvs27>4</hvs27><hvs28>3</hvs28><hvs29>6</hvs29><hvs30>47</hvs30><hvs31>10</hvs31><hvs32>23</h

In [5]:
root = ET.fromstring(response.text)

header = root.find("header")
header_dict = {child.tag: child.text for child in header} if header is not None else {}

items = []
for it in root.findall(".//item"):
    item_dict = {child.tag: child.text for child in it}
    items.append(item_dict)

result = {
    "header": header_dict,
    "items": items,
}

print(json.dumps(result, ensure_ascii=False, indent=2))


{
  "header": {
    "resultCode": "00",
    "resultMsg": "NORMAL SERVICE."
  },
  "items": [
    {
      "dutyName": "연세대학교의과대학강남세브란스병원",
      "dutyTel3": "02-2019-3333",
      "hpid": "A1100015",
      "hv10": "Y",
      "hv11": "Y",
      "hv28": "2",
      "hv29": "1",
      "hv30": "2",
      "hv34": "1",
      "hv41": "4",
      "hv42": "Y",
      "hv5": "Y",
      "hv6": "2",
      "hv7": "Y",
      "hvamyn": "Y",
      "hvangioayn": "Y",
      "hvcrrtayn": "Y",
      "hvctayn": "Y",
      "hvec": "16",
      "hvecmoayn": "Y",
      "hvgc": "86",
      "hvhypoayn": "Y",
      "hvicc": "6",
      "hvidate": "20251129071227",
      "hvincuayn": "Y",
      "hvmriayn": "Y",
      "hvncc": "0",
      "hvoc": "23",
      "hvoxyayn": "N1",
      "hvs01": "21",
      "hvs02": "2",
      "hvs03": "1",
      "hvs04": "2",
      "hvs08": "28",
      "hvs12": "12",
      "hvs15": "8",
      "hvs17": "29",
      "hvs22": "23",
      "hvs25": "5",
      "hvs26": "1",
      "hvs27": "4",
     

In [6]:
def _safe_xml_find(node: Optional[ET.Element], tag: str) -> ET.Element:
    """
    안전한 XML 파싱: node가 None이거나 tag가 없어도 AttributeError 대신
    빈 Element를 반환하여 .text 호출이 항상 안전하도록 함.
    """
    if node is None:
        return ET.Element(tag) # 빈 엘리먼트 반환
    child = node.find(tag)
    if child is None:
        return ET.Element(tag) # 빈 엘리먼트 반환
    return child

def _safe_api_request(endpoint: str, params: Dict[str, str]) -> Dict[str, Any]:
    """
    (P1, P2, P6) requests와 XML 파싱 로직을 래핑합니다.
    API 호출 실패, 타임아웃, XML 파싱 오류, API 결과 오류를 모두 처리하고
    구조화된 dict를 반환합니다.
    """
    try:
        response = requests.get(endpoint, params=params, timeout=10)
        response.raise_for_status()

        root = ET.fromstring(response.content)
        header = _safe_xml_find(root, "header")
        result_code = _safe_xml_find(header, "resultCode").text
        result_msg = _safe_xml_find(header, "resultMsg").text

        if result_code != "00":
            return {"status": "ERROR", "message": f"API 오류: {result_msg} (코드: {result_code})"}

        items_node = _safe_xml_find(_safe_xml_find(root, "body"), "items")
        items = []
        for item_node in items_node.findall("item"):
            item_dict = {child.tag: child.text for child in item_node}
            items.append(item_dict)
        
        if not items:
            return {"status": "OK", "message": "조회된 응급의료기관 정보가 없습니다.", "items": []}
            
        return {"status": "OK", "items": items}

    except requests.exceptions.RequestException as e:
        return {"status": "ERROR", "message": f"네트워크 오류: {str(e)}"}
    except ET.ParseError as e:
        return {"status": "ERROR", "message": f"XML 파싱 오류: {str(e)}"}
    except Exception as e:
        return {"status": "ERROR", "message": f"알 수 없는 오류: {str(e)}"}

# 1. Scenario Design (Single-Trun)

In [7]:
SELECTED_DOMAIN = "재난/응급의료 - 응급실/외상센터 실시간 조회"

print(f"✅ 선택한 도메인: {SELECTED_DOMAIN}")

✅ 선택한 도메인: 재난/응급의료 - 응급실/외상센터 실시간 조회


## 1.0. Define TOOLS

In [8]:
def save_jsonl(items: List[Dict[str, Any]], path: str):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")
    print(f"✅ saved: {path}")


In [9]:
API_URLS = {
    "emergency_list": "https://apis.data.go.kr/B552657/ErmctInfoInqireService/getEmrrmRltmUsefulSckbdInfoInqire",
    "basic_info": "https://apis.data.go.kr/B552657/ErmctInfoInqireService/getEgytListInfoInqire",
    "severe_disease": "https://apis.data.go.kr/B552657/ErmctInfoInqireService/getSrsillDissAceptncPosblInfoInqire",
    "trauma_center":  "https://apis.data.go.kr/B552657/ErmctInfoInqireService/getStrmListInfoInqire",
    "er_message": "https://apis.data.go.kr/B552657/ErmctInfoInqireService/getEmrrmSrsillDissMsgInqire",
}

In [10]:
import requests
import xml.etree.ElementTree as ET
import json
from typing import Any, Dict, List

# ============================================================
# 0. 공공데이터 응급의료 API URL
# ============================================================
SESSION = requests.Session()
DEFAULT_TIMEOUT = 5  # seconds


# ============================================================
# 1. 공통 유틸: XML → items, 안전 호출
# ============================================================

def _parse_xml_items(xml_text: str) -> List[Dict[str, Any]]:
    """
    <item> 태그들만 뽑아서 {tag: text} 딕셔너리 리스트로 변환.
    """
    root = ET.fromstring(xml_text)
    items: List[Dict[str, Any]] = []
    for item in root.iter("item"):
        record: Dict[str, Any] = {}
        for child in item:
            record[child.tag] = (child.text or "").strip()
        items.append(record)
    return items


def _safe_api_request(endpoint_key: str, params: Dict[str, Any]) -> Dict[str, Any]:
    """
    실제 공공데이터 API 호출 + 오류 처리 + XML 파싱.
    리턴 형식 예:
      {"status": "OK", "items": [...]}
      {"status": "ERROR", "message": "..."}
    """
    if endpoint_key not in API_URLS:
        return {"status": "ERROR", "message": f"Unknown endpoint_key: {endpoint_key}"}

    url = API_URLS[endpoint_key]

    # serviceKey는 필수
    if not params.get("serviceKey"):
        return {"status": "ERROR", "message": "SERVICE_KEY가 설정되지 않았습니다."}

    try:
        resp = SESSION.get(url, params=params, timeout=DEFAULT_TIMEOUT)
        resp.raise_for_status()
    except requests.Timeout:
        return {"status": "ERROR", "message": "네트워크 오류: Connection timed out"}
    except requests.RequestException as e:
        return {"status": "ERROR", "message": f"HTTP 오류: {e}"}

    try:
        items = _parse_xml_items(resp.text)
        return {"status": "OK", "items": items}
    except Exception as e:
        return {"status": "ERROR", "message": f"XML 파싱 오류: {e}"}


# ============================================================
# 2. Tool 함수들 (실제 API 호출) - 전부 str(JSON) 리턴
# ============================================================

def get_emergency_room_list(STAGE1: str, STAGE2: str,
                            pageNo: int = 1, numOfRows: int = 20) -> str:
    """
    [Tool 1] (실시간 병상 조회)
    '시도'(STAGE1)와 '시군구'(STAGE2)로 일반 응급실 목록과
    실시간 병상(CT, MRI, 소아 등) 정보를 조회합니다.
    """
    if SERVICE_KEY == "test_key_error":
        return json.dumps(
            {"status": "ERROR",
             "message": "API 오류: SERVICE_KEY_IS_NOT_REGISTERED_ERROR (코드: 30)"},
            ensure_ascii=False,
        )
    if SERVICE_KEY == "test_timeout_error":
        return json.dumps(
            {"status": "ERROR",
             "message": "네트워크 오류: Connection timed out"},
            ensure_ascii=False,
        )

    params = {
        "serviceKey": SERVICE_KEY,
        "STAGE1": STAGE1,
        "STAGE2": STAGE2,
        "pageNo": str(pageNo),
        "numOfRows": str(numOfRows),
    }
    response_dict = _safe_api_request("emergency_list", params)
    return json.dumps(response_dict, ensure_ascii=False)


def get_hospital_basic_info_by_hpid(HPID: str,
                                    pageNo: int = 1, numOfRows: int = 10) -> str:
    """
    [Tool 2] (전문 진료 정보)
    'HPID'(기관ID)로 해당 병원의 '전문 진료'(뇌출혈수술, 사지접합 등) 가능 여부를 조회합니다.
    (공공데이터 문서상 목록/기본정보 API가 여러 개라서,
     여기서는 getEgytListInfoInqire + HPID 필터를 사용.)
    """
    params = {
        "serviceKey": SERVICE_KEY,
        "HPID": HPID,
        "pageNo": str(pageNo),
        "numOfRows": str(numOfRows),
    }
    response_dict = _safe_api_request("basic_info", params)
    return json.dumps(response_dict, ensure_ascii=False)


def get_severe_disease_info(STAGE1: str, STAGE2: str,
                            pageNo: int = 1, numOfRows: int = 20) -> str:
    """
    [Tool 3] (중증질환 수용 가능 정보)
    '시도'(STAGE1)와 '시군구'(STAGE2)로 중증질환 수용 가능 정보를 조회합니다.
    (getSrsillDissAceptncPosblInfoInqire)
    """
    params = {
        "serviceKey": SERVICE_KEY,
        "STAGE1": STAGE1,
        "STAGE2": STAGE2,
        "pageNo": str(pageNo),
        "numOfRows": str(numOfRows),
    }
    response_dict = _safe_api_request("severe_disease", params)
    return json.dumps(response_dict, ensure_ascii=False)


def get_trauma_center_list(Q0: str, Q1: str,
                           pageNo: int = 1, numOfRows: int = 20) -> str:
    """
    [Tool 4] (외상센터 목록)
    '시도'(Q0)와 '시군구'(Q1)로 외상센터 목록을 조회합니다.
    (getTrtmListInfoInqire, 공공데이터 예시에서는 Q0/Q1 사용)
    """
    params = {
        "serviceKey": SERVICE_KEY,
        "Q0": Q0,
        "Q1": Q1,
        "pageNo": str(pageNo),
        "numOfRows": str(numOfRows),
    }
    response_dict = _safe_api_request("trauma_center", params)
    return json.dumps(response_dict, ensure_ascii=False)


def get_er_message_by_hpid(HPID: str,
                           pageNo: int = 1, numOfRows: int = 10) -> str:
    """
    [Tool 5] (응급실/중증질환 메시지)
    'HPID'(기관ID)로 해당 응급실의 실시간 공지 메시지
    (예: 특정 중증질환 '차단' 여부 등)를 조회합니다.
    (getEmrrmSrsillDissMsgInqire)
    """
    params = {
        "serviceKey": SERVICE_KEY,
        "HPID": HPID,
        "pageNo": str(pageNo),
        "numOfRows": str(numOfRows),
    }
    response_dict = _safe_api_request("er_message", params)
    return json.dumps(response_dict, ensure_ascii=False)


def get_travel_time(origin: str, destination: str) -> str:
    """
    [Tool 6] (가상 거리)
    출발지와 목적지 간의 예상 이동 시간(분)을 조회합니다.
    → 이건 공공데이터 API가 아니라, 시나리오용 가상 Tool로 남겨둠.
    """
    return json.dumps(
        {
            "status": "MOCK",
            "origin": origin,
            "destination": destination,
            "estimated_minutes": 18,
        },
        ensure_ascii=False,
    )


# ============================================================
# 3. Function Calling용 TOOLS 정의 (Solar/OpenAI tools 스펙)
# ============================================================

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_emergency_room_list",
            "description": "[Tool 1] (일반 응급실/실시간 병상) '시도'와 '시군구'로 일반 응급실 목록과 실시간 병상(CT, MRI, 소아 등) 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "STAGE1": {
                        "type": "string",
                        "description": "주소(시도) (예: '서울특별시')",
                    },
                    "STAGE2": {
                        "type": "string",
                        "description": "주소(시군구) (예: '성북구')",
                    },
                    "pageNo": {
                        "type": "integer",
                        "minimum": 1,
                        "default": 1,
                    },
                    "numOfRows": {
                        "type": "integer",
                        "minimum": 1,
                        "maximum": 100,
                        "default": 20,
                    },
                },
                "required": ["STAGE1", "STAGE2"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_hospital_basic_info_by_hpid",
            "description": "[Tool 2] (전문 진료 정보) 'HPID'(기관ID)로 해당 병원의 '전문 진료'(뇌출혈수술, 사지접합 등) 가능 여부를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "HPID": {
                        "type": "string",
                        "description": "병원 기관ID (예: 'A1100008')",
                    },
                    "pageNo": {
                        "type": "integer",
                        "minimum": 1,
                        "default": 1,
                    },
                    "numOfRows": {
                        "type": "integer",
                        "minimum": 1,
                        "maximum": 100,
                        "default": 10,
                    },
                },
                "required": ["HPID"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_severe_disease_info",
            "description": "[Tool 3] (중증질환 수용) '시도'와 '시군구'로 중증질환 수용 가능 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "STAGE1": {
                        "type": "string",
                        "description": "주소(시도) (예: '서울특별시')",
                    },
                    "STAGE2": {
                        "type": "string",
                        "description": "주소(시군구) (예: '강북구')",
                    },
                    "pageNo": {
                        "type": "integer",
                        "minimum": 1,
                        "default": 1,
                    },
                    "numOfRows": {
                        "type": "integer",
                        "minimum": 1,
                        "maximum": 100,
                        "default": 20,
                    },
                },
                "required": ["STAGE1", "STAGE2"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_trauma_center_list",
            "description": "[Tool 4] (외상센터) '시도'(Q0)와 '시군구'(Q1)로 '외상센터' 목록을 조회합니다. (교통사고, 추락 등 중증 외상 환자용)",
            "parameters": {
                "type": "object",
                "properties": {
                    "Q0": {
                        "type": "string",
                        "description": "주소(시도) (예: '서울특별시')",
                    },
                    "Q1": {
                        "type": "string",
                        "description": "주소(시군구) (예: '도봉구')",
                    },
                    "pageNo": {
                        "type": "integer",
                        "minimum": 1,
                        "default": 1,
                    },
                    "numOfRows": {
                        "type": "integer",
                        "minimum": 1,
                        "maximum": 100,
                        "default": 20,
                    },
                },
                "required": ["Q0", "Q1"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_er_message_by_hpid",
            "description": "[Tool 5] (응급실 메시지) 'HPID'(기관ID)로 해당 응급실의 '실시간 공지 메시지'(예: 특정 중증질환 '차단' 여부)를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "HPID": {
                        "type": "string",
                        "description": "병원 기관ID (예: 'A1100008')",
                    },
                    "pageNo": {
                        "type": "integer",
                        "minimum": 1,
                        "default": 1,
                    },
                    "numOfRows": {
                        "type": "integer",
                        "minimum": 1,
                        "maximum": 100,
                        "default": 10,
                    },
                },
                "required": ["HPID"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_travel_time",
            "description": "[Tool 6] (가상 거리) 출발지와 목적지 간의 예상 이동 시간(분)을 조회합니다. (시나리오용 가상 Tool)",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {
                        "type": "string",
                        "description": "출발지 주소 또는 장소명 (예: '도봉역')",
                    },
                    "destination": {
                        "type": "string",
                        "description": "목적지 주소 또는 장소명 (예: '고려대학교안암병원')",
                    },
                },
                "required": ["origin", "destination"],
            },
        },
    },
]

In [11]:
SELECTED_DOMAIN = "재난/응급의료 - 응급실/외상센터 실시간 조회"

print("=" * 60)
print("✅ 응급의료 TOOLS 정의 완료!")
print("=" * 60)
print(f"선택한 도메인: {SELECTED_DOMAIN}")
print(f"정의된 Tool 개수: {len(TOOLS)}개")
for i, tool in enumerate(TOOLS, 1):
    fn = tool["function"]
    print(f"  {i}. {fn['name']}  -  {fn['description']}")
print("=" * 60)

# SERVICE_KEY 없으면 실 API 호출은 스킵
if ("SERVICE_KEY" not in globals()) or (not SERVICE_KEY) or (SERVICE_KEY.startswith("YOUR_")):
    print("\n⚠ SERVICE_KEY가 설정되어 있지 않아서, 실 공공데이터 API 호출 테스트는 건너뜁니다.")
    print("   → SERVICE_KEY를 실제 키로 설정한 뒤 다시 실행하세요.\n")
else:
    print("\n✅ SERVICE_KEY 설정 확인! 실 공공데이터 API 호출 테스트를 수행합니다.\n")

    # ------------------------------------------------------------
    # 테스트 1: get_emergency_room_list (실시간 응급실/병상)
    # ------------------------------------------------------------
    print("[테스트 1] get_emergency_room_list(STAGE1='서울특별시', STAGE2='성북구')")

    try:
        res_str = get_emergency_room_list(STAGE1="서울특별시", STAGE2="성북구")
        print("  ▸ 타입:", type(res_str).__name__)
        data = json.loads(res_str)
        print("  ▸ JSON 파싱 성공, 최상위 키들:", list(data.keys())[:8])
        if data.get("status") == "OK":
            items = data.get("items", [])
            print("  ▸ 조회된 응급실 수:", len(items))
            if items:
                sample = items[0]
                print("  ▸ 예시 병원명(dutyName):", sample.get("dutyName"))
                print("  ▸ 예시 주소(dutyAddr):", sample.get("dutyAddr"))
        else:
            print("  ⚠ API 응답 status:", data.get("status"), "| message:", data.get("message"))
    except Exception as e:
        print("  ❌ 오류 발생:", e)

    # ------------------------------------------------------------
    # 테스트 2: get_hospital_basic_info_by_hpid (전문 진료/기본 정보)
    # ------------------------------------------------------------
    print("\n[테스트 2] get_hospital_basic_info_by_hpid(HPID='A1100008')")

    try:
        res_str = get_hospital_basic_info_by_hpid(HPID="A1100008")
        print("  ▸ 타입:", type(res_str).__name__)
        data = json.loads(res_str)
        print("  ▸ JSON 파싱 성공, 최상위 키들:", list(data.keys())[:8])
        if data.get("status") == "OK":
            items = data.get("items", [])
            print("  ▸ 조회된 레코드 수:", len(items))
            if items:
                sample = items[0]
                print("  ▸ 예시 병원명(dutyName):", sample.get("dutyName"))
                print("  ▸ 샘플 필드들:", list(sample.keys())[:10])
        else:
            print("  ⚠ API 응답 status:", data.get("status"), "| message:", data.get("message"))
    except Exception as e:
        print("  ❌ 오류 발생:", e)

    # ------------------------------------------------------------
    # 테스트 3: get_severe_disease_info (중증질환 수용 가능 정보)
    # ------------------------------------------------------------
    print("\n[테스트 3] get_severe_disease_info(STAGE1='서울특별시', STAGE2='강북구')")

    try:
        res_str = get_severe_disease_info(STAGE1="서울특별시", STAGE2="강북구")
        print("  ▸ 타입:", type(res_str).__name__)
        data = json.loads(res_str)
        print("  ▸ JSON 파싱 성공, 최상위 키들:", list(data.keys())[:8])
        if data.get("status") == "OK":
            items = data.get("items", [])
            print("  ▸ 조회된 레코드 수:", len(items))
            if items:
                sample = items[0]
                print("  ▸ 예시 병원명(dutyName):", sample.get("dutyName"))
                print("  ▸ 샘플 필드들:", list(sample.keys())[:10])
        else:
            print("  ⚠ API 응답 status:", data.get("status"), "| message:", data.get("message"))
    except Exception as e:
        print("  ❌ 오류 발생:", e)

    # ------------------------------------------------------------
    # 테스트 4: get_trauma_center_list (외상센터 목록)
    # ------------------------------------------------------------
    print("\n[테스트 4] get_trauma_center_list(Q0='서울특별시', Q1='도봉구')")

    try:
        res_str = get_trauma_center_list(Q0="서울특별시", Q1="도봉구")
        print("  ▸ 타입:", type(res_str).__name__)
        data = json.loads(res_str)
        print("  ▸ JSON 파싱 성공, 최상위 키들:", list(data.keys())[:8])
        if data.get("status") == "OK":
            items = data.get("items", [])
            print("  ▸ 조회된 외상센터 수:", len(items))
            if items:
                sample = items[0]
                print("  ▸ 예시 센터명(dutyName 등):", sample.get("dutyName"))
                print("  ▸ 샘플 필드들:", list(sample.keys())[:10])
        else:
            print("  ⚠ API 응답 status:", data.get("status"), "| message:", data.get("message"))
    except Exception as e:
        print("  ❌ 오류 발생:", e)

    # ------------------------------------------------------------
    # 테스트 5: get_er_message_by_hpid (응급실 메시지)
    # ------------------------------------------------------------
    print("\n[테스트 5] get_er_message_by_hpid(HPID='A1100008')")

    try:
        res_str = get_er_message_by_hpid(HPID="A1100008")
        print("  ▸ 타입:", type(res_str).__name__)
        data = json.loads(res_str)
        print("  ▸ JSON 파싱 성공, 최상위 키들:", list(data.keys())[:8])
        if data.get("status") == "OK":
            items = data.get("items", [])
            print("  ▸ 조회된 메시지 수:", len(items))
            if items:
                sample = items[0]
                # 실제 필드는 응답 보고 맞춰야 함 (ex: msg, dutyName 등)
                print("  ▸ 예시 병원명(dutyName):", sample.get("dutyName"))
                print("  ▸ 샘플 필드들:", list(sample.keys())[:10])
        else:
            print("  ⚠ API 응답 status:", data.get("status"), "| message:", data.get("message"))
    except Exception as e:
        print("  ❌ 오류 발생:", e)

# ------------------------------------------------------------
# 테스트 6: get_travel_time (가상 Tool - 여전히 MOCK)
#   ※ 이건 공공데이터 API가 아니라 길찾기 API 필요해서,
#      지금은 시나리오용 가상 응답 유지하는 게 맞음.
# ------------------------------------------------------------
print("\n[테스트 6] get_travel_time(origin='도봉역', destination='고려대학교안암병원')  # 시나리오용 가상 Tool")

try:
    res_str = get_travel_time(origin="도봉역", destination="고려대학교안암병원")
    print("  ▸ 타입:", type(res_str).__name__)
    data = json.loads(res_str)
    print("  ▸ JSON 파싱 성공, 내용:", data)
except Exception as e:
    print("  ❌ 오류 발생:", e)

print("\n" + "=" * 60)
print("✅ 모든 Tool 함수의 기본 동작 및 JSON 직렬화(P5) 확인 완료 (또는 위 오류 메시지 참고)")
print("=" * 60)


✅ 응급의료 TOOLS 정의 완료!
선택한 도메인: 재난/응급의료 - 응급실/외상센터 실시간 조회
정의된 Tool 개수: 6개
  1. get_emergency_room_list  -  [Tool 1] (일반 응급실/실시간 병상) '시도'와 '시군구'로 일반 응급실 목록과 실시간 병상(CT, MRI, 소아 등) 정보를 조회합니다.
  2. get_hospital_basic_info_by_hpid  -  [Tool 2] (전문 진료 정보) 'HPID'(기관ID)로 해당 병원의 '전문 진료'(뇌출혈수술, 사지접합 등) 가능 여부를 조회합니다.
  3. get_severe_disease_info  -  [Tool 3] (중증질환 수용) '시도'와 '시군구'로 중증질환 수용 가능 정보를 조회합니다.
  4. get_trauma_center_list  -  [Tool 4] (외상센터) '시도'(Q0)와 '시군구'(Q1)로 '외상센터' 목록을 조회합니다. (교통사고, 추락 등 중증 외상 환자용)
  5. get_er_message_by_hpid  -  [Tool 5] (응급실 메시지) 'HPID'(기관ID)로 해당 응급실의 '실시간 공지 메시지'(예: 특정 중증질환 '차단' 여부)를 조회합니다.
  6. get_travel_time  -  [Tool 6] (가상 거리) 출발지와 목적지 간의 예상 이동 시간(분)을 조회합니다. (시나리오용 가상 Tool)

✅ SERVICE_KEY 설정 확인! 실 공공데이터 API 호출 테스트를 수행합니다.

[테스트 1] get_emergency_room_list(STAGE1='서울특별시', STAGE2='성북구')
  ▸ 타입: str
  ▸ JSON 파싱 성공, 최상위 키들: ['status', 'items']
  ▸ 조회된 응급실 수: 1
  ▸ 예시 병원명(dutyName): 학교법인고려중앙학원고려대학교의과대학부속병원(안암병원)
  ▸ 예시 주소(dutyAddr): None

[테스트 2] get_hospital_basic_inf

## 1-1. Scenario Design 1 (Single-Turn)

In [12]:
# ======================================================================================
# 🏥 폭설 상황 싱글턴 시나리오 (Solar-Pro2 + 공공데이터 API + BFCL 포맷 저장 + 프롬프트 증강)
# ======================================================================================
import os
import json
import random
import requests
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Optional
from openai import OpenAI

# --------------------------------------------------
# 0. 공용 유틸
# --------------------------------------------------
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
DEBUG_LOG_FILE = OUTPUT_DIR / "debug_log_snow_single.txt"


def debug_log(msg: str, level: str = "INFO") -> None:
    line = f"[{datetime.now().strftime('%H:%M:%S')}] [{level}] {msg}"
    print(line)
    with open(DEBUG_LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def save_jsonl(items: List[Dict[str, Any]], path: str):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")
    print(f"✅ saved: {path}")


# ======================================================================================
# 🔹 A. 프롬프트 증강 유틸 (원본 → 여러 개 변형 프롬프트)
# ======================================================================================

random.seed(42)  # 재현성 확보

SYNONYM_MAP = {
    # 상황 / 날씨
    "폭설": ["폭설 상황", "눈이 많이 오는 상황", "대설"],
    "도로": ["도로 상황", "길", "도로 상태"],
    # 지역
    "성북구": ["성북구 일대", "서울 성북구"],
    "도봉구": ["도봉구 일대", "서울 도봉구"],
    "강북구": ["강북구 일대", "서울 강북구"],
    # 의료/응급
    "응급실": ["응급센터", "응급 진료실", "응급실(ER)"],
    "병원": ["의료기관", "병원 기관"],
    "병상": ["입원 병상", "병상 수"],
    "수용 불가": ["수용이 어려운", "진료가 어려운", "받기 힘든"],
    "정상 운영": ["정상으로 운영 중인", "현재 운영 중인", "지금 열려 있는"],
}


def random_deletion(words, p=0.08):
    """단어들을 확률 p로 랜덤 삭제"""
    if len(words) == 1:
        return words
    new_words = [w for w in words if random.uniform(0, 1) > p]
    if not new_words:
        new_words.append(random.choice(words))
    return new_words


def random_swap(words, n_swaps=1):
    """문장 내 단어 위치를 n_swaps번 랜덤으로 교환"""
    if len(words) < 2:
        return words
    new_words = words[:]
    for _ in range(n_swaps):
        i, j = random.sample(range(len(new_words)), 2)
        new_words[i], new_words[j] = new_words[j], new_words[i]
    return new_words


def synonym_replacement(text: str, replace_prob: float = 0.4) -> str:
    """
    SYNONYM_MAP 기반 간단 유의어 치환
    """
    new_text = text
    for key, syn_list in SYNONYM_MAP.items():
        if key in new_text and random.random() < replace_prob:
            synonym = random.choice(syn_list)
            new_text = new_text.replace(key, synonym)
    return new_text


def augment_text(
    text: str,
    alpha_del: float = 0.08,
    max_swaps: int = 2,
    synonym_prob: float = 0.4,
) -> str:
    """
    1) 유의어 치환 → 2) 약간의 단어 삭제 → 3) 약간의 단어 위치 스왑
    """
    # 1) 유의어 치환
    text = synonym_replacement(text, replace_prob=synonym_prob)

    words = text.split()
    if len(words) <= 3:
        return text

    # 2) 삭제
    words = random_deletion(words, p=alpha_del)

    # 3) 스왑
    n_swaps = min(max_swaps, max(1, len(words) // 12))
    words = random_swap(words, n_swaps=n_swaps)

    return " ".join(words)


# ======================================================================================
# 3. Tools (BFCL 호환 스키마)
#    (여기 위/아래로 SOLAR_API_KEY, SERVICE_KEY, API_URLS, client 정의는
#     네가 기존에 쓰던 그대로 두면 됨)
# ======================================================================================
TOOLS: List[Dict[str, Any]] = [
    {
        "type": "function",
        "function": {
            "name": "get_emergency_room_list",
            "description": "시/도·시군구 단위로 실시간 응급실 가용 병상 및 운영 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "STAGE1": {
                        "type": "string",
                        "description": "시/도명 (예: 서울특별시)",
                    },
                    "STAGE2": {
                        "type": "string",
                        "description": "시/군/구명 (예: 성북구, 도봉구, 강북구)",
                    },
                },
                "required": ["STAGE1", "STAGE2"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_er_message_by_hpid",
            "description": "HPID로 해당 병원의 응급실 실시간 메시지(임시 폐쇄, 수용 불가 등)를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "HPID": {
                        "type": "string",
                        "description": "병원 ID (예: A1100008)",
                    }
                },
                "required": ["HPID"],
            },
        },
    },
]

# --------------------------------------------------
# 4. 도구 실행 함수: Solar가 호출한 tool을 실제 API로 연결
#    (SERVICE_KEY, API_URLS, client 등은 네가 기존에 선언한 그대로 사용)
# --------------------------------------------------
def execute_tool_call(name: str, args: Dict[str, Any]) -> str:
    debug_log(f"🛠️ execute_tool_call: {name} | args={args}")

    try:
        if name == "get_emergency_room_list":
            params = {
                "serviceKey": SERVICE_KEY,
                "STAGE1": args["STAGE1"],
                "STAGE2": args["STAGE2"],
                "pageNo": "1",
                "numOfRows": "100",
            }
            res = requests.get(API_URLS["emergency_list"], params=params, timeout=5)
            res.raise_for_status()
            return json.dumps(
                {
                    "status": "OK",
                    "api": "getEmrrmRltmUsefulSckbdInfoInqire",
                    "stage1": args["STAGE1"],
                    "stage2": args["STAGE2"],
                    "raw_xml": res.text,
                },
                ensure_ascii=False,
            )

        elif name == "get_er_message_by_hpid":
            params = {
                "serviceKey": SERVICE_KEY,
                "HPID": args["HPID"],
                "pageNo": "1",
                "numOfRows": "20",
            }
            res = requests.get(API_URLS["er_message"], params=params, timeout=5)
            res.raise_for_status()
            return json.dumps(
                {
                    "status": "OK",
                    "api": "getEmrrmSrsillDissMsgInqire",
                    "hpid": args["HPID"],
                    "raw_xml": res.text,
                },
                ensure_ascii=False,
            )

        else:
            return json.dumps(
                {"status": "ERROR", "message": f"Unknown tool: {name}"},
                ensure_ascii=False,
            )

    except Exception as e:
        debug_log(f"❌ tool 실행 실패: {e}", level="ERROR")
        return json.dumps(
            {"status": "ERROR", "message": f"{type(e).__name__}: {e}"},
            ensure_ascii=False,
        )


# --------------------------------------------------
# 5. 싱글턴 시나리오 생성 (프롬프트를 인자로 받아서 사용)
# --------------------------------------------------
def generate_snow_single_scenario(
    user_text: str,
    scenario_id: str,
    is_augmented: bool = False,
    aug_index: int = 0,
    base_scenario_id: Optional[str] = None,
) -> Optional[Dict[str, Any]]:
    """
    user_text: 실제 Solar에 던질 유저 프롬프트 (원본/증강 둘 다 가능)
    scenario_id: 이 row에 부여할 id (예: single_snow_01, single_snow_01_aug1)
    """

    # BFCL-style 메시지 로그
    conversation: List[Dict[str, Any]] = []

    # 1) system + user
    system_msg = {
        "role": "system",
        "content": (
            "당신은 한국의 응급의료 상황을 도와주는 AI 비서입니다. "
            "사용자의 질문에 답변할 때는 반드시 제공된 도구를 사용해 "
            "실시간 정보를 조회한 후, 그 결과를 바탕으로 답변해야 합니다."
        ),
    }
    user_msg = {"role": "user", "content": user_text}

    conversation.append(system_msg)
    conversation.append(user_msg)

    messages_for_solar = [system_msg, user_msg]

    # 2) 첫 번째 호출: 반드시 tool_call 생성
    try:
        debug_log(f"🚀 Solar 1차 호출 (tool_choice=required) | {scenario_id}")
        first = client.chat.completions.create(
            model="solar-pro2",
            messages=messages_for_solar,
            tools=TOOLS,
            tool_choice="required",
        )
    except Exception as e:
        debug_log(f"Solar 1차 호출 실패: {e}", level="ERROR")
        return None

    ai_msg = first.choices[0].message

    tool_calls_list: List[Dict[str, Any]] = []
    if ai_msg.tool_calls:
        debug_log(f"✅ tool_calls 개수: {len(ai_msg.tool_calls)}")
        for tc in ai_msg.tool_calls:
            tool_calls_list.append(
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
            )
    else:
        debug_log("⚠️ tool_calls 없음 (원래는 나오도록 설계돼야 함)", level="WARN")

    assistant_with_tools_msg = {
        "role": "assistant",
        "content": ai_msg.content or "",
        "tool_calls": tool_calls_list if tool_calls_list else [],
    }
    conversation.append(assistant_with_tools_msg)
    messages_for_solar.append(ai_msg)

    # 3) tool_calls 실제 실행 후, tool 메시지들 추가
    if ai_msg.tool_calls:
        for tc in ai_msg.tool_calls:
            try:
                args = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                debug_log(
                    f"❌ arguments JSON 파싱 실패: {tc.function.arguments}",
                    level="ERROR",
                )
                continue

            tool_result = execute_tool_call(tc.function.name, args)

            tool_msg = {
                "role": "tool",
                "tool_call_id": tc.id,
                "content": tool_result,
            }
            conversation.append(tool_msg)
            messages_for_solar.append(tool_msg)

    # 4) 두 번째 호출: 최종 자연어 답변
    try:
        debug_log(f"🚀 Solar 2차 호출 (최종 답변) | {scenario_id}")
        second = client.chat.completions.create(
            model="solar-pro2",
            messages=messages_for_solar,
            tools=TOOLS,
        )
    except Exception as e:
        debug_log(f"Solar 2차 호출 실패: {e}", level="ERROR")
        return None

    final_ai_msg = second.choices[0].message
    final_assistant_msg = {
        "role": "assistant",
        "content": final_ai_msg.content or "",
    }
    conversation.append(final_assistant_msg)

    # 5) metadata 구성 (증강 여부 표시)
    if base_scenario_id is None:
        base_scenario_id = scenario_id

    metadata_obj = {
        "tools": TOOLS,
        "scenario_meta": {
            "type": "Single-turn",
            "task": "폭설 상황에서 성북/도봉/강북구 정상 운영 응급실 필터링",
            "augmentation": {
                "is_augmented": is_augmented,
                "aug_index": aug_index if is_augmented else 0,
                "base_scenario_id": base_scenario_id,
            },
        },
    }

    row = {
        "id": scenario_id,
        "row": {
            "messages": conversation,
            "metadata": json.dumps(metadata_obj, ensure_ascii=False),
        },
    }

    return row


# --------------------------------------------------
# 6. 메인: 원본 + 증강 프롬프트들에 대해 시나리오 생성
# --------------------------------------------------
if __name__ == "__main__":
    debug_log("==== 폭설 싱글턴 시나리오 (원본 + 증강) 생성 시작 ====")

    base_scenario_id = "single_snow_01"
    base_user_text = (
        "폭설 때문에 도로가 많이 막힌 상황인데, "
        "서울 성북구/도봉구/강북구 쪽에서 지금 정상 운영 중인 응급실만 알려줘. "
        "임시 폐쇄되었거나 수용 불가인 곳은 빼줘."
    )

    N_AUG = 3  # 원본 1개 + 증강 3개 → 총 4개 시나리오

    rows: List[Dict[str, Any]] = []

    # 1) 원본 프롬프트 시나리오
    base_row = generate_snow_single_scenario(
        user_text=base_user_text,
        scenario_id=base_scenario_id,
        is_augmented=False,
    )
    if base_row:
        rows.append(base_row)

    # 2) 증강 프롬프트 시나리오들
    for i in range(1, N_AUG + 1):
        aug_text = augment_text(base_user_text)
        aug_id = f"{base_scenario_id}_aug{i}"

        debug_log(f"--- 증강 프롬프트 {i}: {aug_text}")

        aug_row = generate_snow_single_scenario(
            user_text=aug_text,
            scenario_id=aug_id,
            is_augmented=True,
            aug_index=i,
            base_scenario_id=base_scenario_id,
        )
        if aug_row:
            rows.append(aug_row)

    if not rows:
        debug_log("시나리오 생성 실패 (row 없음)", level="ERROR")
    else:
        out_path = str(OUTPUT_DIR / "snow_single_turn_dataset_with_aug.jsonl")
        save_jsonl(rows, out_path)
        debug_log("완료 ✅")


[07:20:36] [INFO] ==== 폭설 싱글턴 시나리오 (원본 + 증강) 생성 시작 ====
[07:20:36] [INFO] 🚀 Solar 1차 호출 (tool_choice=required) | single_snow_01
[07:20:38] [INFO] ✅ tool_calls 개수: 3
[07:20:38] [INFO] 🛠️ execute_tool_call: get_emergency_room_list | args={'STAGE1': '서울특별시', 'STAGE2': '성북구'}
[07:20:38] [INFO] 🛠️ execute_tool_call: get_emergency_room_list | args={'STAGE1': '서울특별시', 'STAGE2': '도봉구'}
[07:20:38] [INFO] 🛠️ execute_tool_call: get_emergency_room_list | args={'STAGE1': '서울특별시', 'STAGE2': '강북구'}
[07:20:38] [INFO] 🚀 Solar 2차 호출 (최종 답변) | single_snow_01
[07:20:43] [INFO] --- 증강 프롬프트 1: 폭설 때문에 중인 막힌 상황인데, 서울 성북구 일대/도봉구/강북구 지금 정상으로 운영 중인 많이 응급실만 알려줘. 임시 폐쇄되었거나 진료가 어려운인 곳은 빼줘.
[07:20:43] [INFO] 🚀 Solar 1차 호출 (tool_choice=required) | single_snow_01_aug1
[07:20:45] [INFO] ✅ tool_calls 개수: 3
[07:20:45] [INFO] 🛠️ execute_tool_call: get_emergency_room_list | args={'STAGE1': '서울특별시', 'STAGE2': '성북구'}
[07:20:45] [INFO] 🛠️ execute_tool_call: get_emergency_room_list | args={'STAGE1': '서울특별시', 'STAGE2': '도봉구'}
[

## 1-2. Scenario Design 2 (Single-Turn)

In [13]:
import json
import random
import copy
from typing import Any, Dict, List, Optional

# ... (SOLAR_API_KEY, SERVICE_KEY, API_URLS, client, debug_log, save_jsonl 등 기존 코드)

# ============================================================
# A. 프롬프트 증강 유틸
# ============================================================

random.seed(42)  # 재현성

SYNONYM_MAP = {
    # 상황 / 날씨
    "폭설": ["폭설 상황", "눈이 많이 오는 상황", "대설"],
    "도로": ["도로 상황", "길", "도로 상태"],
    # 지역
    "성북구": ["성북구 일대", "서울 성북구"],
    "도봉구": ["도봉구 일대", "서울 도봉구"],
    "강북구": ["강북구 일대", "서울 강북구"],
    "강남구": ["강남구 일대", "서울 강남구"],
    # 응급/의료
    "응급실": ["응급센터", "응급 진료실", "응급실(ER)"],
    "병상": ["입원 병상", "병상 수"],
    "정상 운영": ["현재 정상 운영 중인", "지금 운영 중인", "지금 열려 있는"],
    "수용 불가": ["수용이 어려운", "진료가 어려운", "받기 힘든"],
    "중증질환": ["중증 질환", "심각한 질환"],
}

def random_deletion(words, p=0.08):
    if len(words) == 1:
        return words
    new_words = [w for w in words if random.uniform(0, 1) > p]
    if not new_words:
        new_words.append(random.choice(words))
    return new_words

def random_swap(words, n_swaps=1):
    if len(words) < 2:
        return words
    new_words = words[:]
    for _ in range(n_swaps):
        i, j = random.sample(range(len(new_words)), 2)
        new_words[i], new_words[j] = new_words[j], new_words[i]
    return new_words

def synonym_replacement(text: str, replace_prob: float = 0.4) -> str:
    new_text = text
    for key, syn_list in SYNONYM_MAP.items():
        if key in new_text and random.random() < replace_prob:
            synonym = random.choice(syn_list)
            new_text = new_text.replace(key, synonym)
    return new_text

def augment_text(
    text: str,
    alpha_del: float = 0.08,
    max_swaps: int = 2,
    synonym_prob: float = 0.4,
) -> str:
    # 1) 유의어 치환
    text = synonym_replacement(text, replace_prob=synonym_prob)

    # 2) 단어 삭제/스왑
    words = text.split()
    if len(words) <= 3:
        return text

    words = random_deletion(words, p=alpha_del)
    n_swaps = min(max_swaps, max(1, len(words) // 12))
    words = random_swap(words, n_swaps=n_swaps)
    return " ".join(words)


# ============================================================
# B. tool 실행 함수 (네가 준 그대로)
# ============================================================

def execute_tool_call(name: str, args: Dict[str, Any]) -> str:
    """
    Solar가 만든 tool_call을 실제 공공데이터 API로 호출하고,
    결과를 JSON string으로 반환 (tool message content에 그대로 들어감).
    """
    debug_log(f"🛠️ execute_tool_call: {name} | args={args}")

    try:
        if name == "get_emergency_room_list":
            params = {
                "serviceKey": SERVICE_KEY,
                "STAGE1": args["STAGE1"],
                "STAGE2": args["STAGE2"],
                "pageNo": "1",
                "numOfRows": "100",
            }
            res = requests.get(API_URLS["emergency_list"], params=params, timeout=5)
            res.raise_for_status()
            return json.dumps(
                {
                    "status": "OK",
                    "api": "getEmrrmRltmUsefulSckbdInfoInqire",
                    "stage1": args["STAGE1"],
                    "stage2": args["STAGE2"],
                    "raw_xml": res.text,
                },
                ensure_ascii=False,
            )

        elif name == "get_er_message_by_hpid":
            params = {
                "serviceKey": SERVICE_KEY,
                "HPID": args["HPID"],
                "pageNo": "1",
                "numOfRows": "20",
            }
            res = requests.get(API_URLS["er_message"], params=params, timeout=5)
            res.raise_for_status()
            return json.dumps(
                {
                    "status": "OK",
                    "api": "getEmrrmSrsillDissMsgInqire",
                    "hpid": args["HPID"],
                    "raw_xml": res.text,
                },
                ensure_ascii=False,
            )

        else:
            return json.dumps(
                {"status": "ERROR", "message": f"Unknown tool: {name}"},
                ensure_ascii=False,
            )

    except Exception as e:
        debug_log(f"❌ tool 실행 실패: {e}", level="ERROR")
        return json.dumps(
            {"status": "ERROR", "message": f"{type(e).__name__}: {e}"},
            ensure_ascii=False,
        )


# ============================================================
# C. 싱글턴 시나리오 생성기 (그대로 사용)
# ============================================================

def generate_single_scenario(
    scenario_id: str,
    user_content: str,
    scenario_meta: Dict[str, Any],
) -> Optional[Dict[str, Any]]:
    """
    - system 프롬프트 + user
    - Solar 1차 호출(tool_choice='required') → tool_calls
    - 우리 코드에서 tool_calls 실행 → tool 메시지 추가
    - Solar 2차 호출 → 최종 assistant 답변
    - BFCL/evaluate.py에서 쓰기 좋은 row 구조로 리턴
    """

    conversation: List[Dict[str, Any]] = []

    system_msg = {
        "role": "system",
        "content": (
            "당신은 한국의 응급의료 상황을 도와주는 AI 비서입니다. "
            "반드시 제공된 도구를 사용해 실시간 정보를 조회한 뒤, 그 결과를 근거로 답변하세요."
        ),
    }
    user_msg = {"role": "user", "content": user_content}

    conversation.append(user_msg)
    messages_for_solar: List[Dict[str, Any]] = [system_msg, user_msg]

    try:
        debug_log(f"🚀 Solar 1차 호출 ({scenario_id})")
        first = client.chat.completions.create(
            model="solar-pro2",
            messages=messages_for_solar,
            tools=TOOLS,
            tool_choice="required",
        )
    except Exception as e:
        debug_log(f"Solar 1차 호출 실패: {e}", level="ERROR")
        return None

    ai_msg = first.choices[0].message

    tool_calls_list: List[Dict[str, Any]] = []
    if ai_msg.tool_calls:
        debug_log(f"✅ tool_calls 개수: {len(ai_msg.tool_calls)}")
        for tc in ai_msg.tool_calls:
            tool_calls_list.append(
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
            )
    else:
        debug_log("⚠️ tool_calls 없음", level="WARN")

    assistant_with_tools = {
        "role": "assistant",
        "content": ai_msg.content or "",
        "tool_calls": tool_calls_list if tool_calls_list else [],
    }
    conversation.append(assistant_with_tools)
    messages_for_solar.append(ai_msg)

    if ai_msg.tool_calls:
        for tc in ai_msg.tool_calls:
            try:
                args = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                debug_log(
                    f"❌ arguments JSON 파싱 실패: {tc.function.arguments}",
                    level="ERROR",
                )
                continue

            tool_result = execute_tool_call(tc.function.name, args)

            tool_msg = {
                "role": "tool",
                "tool_call_id": tc.id,
                "content": tool_result,
            }
            conversation.append(tool_msg)
            messages_for_solar.append(tool_msg)

    try:
        debug_log(f"🚀 Solar 2차 호출 ({scenario_id}) - 최종 답변")
        second = client.chat.completions.create(
            model="solar-pro2",
            messages=messages_for_solar,
            tools=TOOLS,
        )
    except Exception as e:
        debug_log(f"Solar 2차 호출 실패: {e}", level="ERROR")
        return None

    final_ai_msg = second.choices[0].message
    final_assistant = {
        "role": "assistant",
        "content": final_ai_msg.content or "",
    }
    conversation.append(final_assistant)

    print("\n" + "=" * 60)
    print(f"[{scenario_id}] 최종 답변:")
    print(final_assistant["content"])
    print("=" * 60 + "\n")

    metadata_obj = {
        "tools": TOOLS,
        "scenario_meta": scenario_meta,
    }

    row = {
        "id": scenario_id,
        "row": {
            "messages": conversation,
            "metadata": json.dumps(metadata_obj, ensure_ascii=False),
        },
    }
    return row


# ============================================================
# D. 시나리오 정의 + 증강 적용
# ============================================================

SCENARIOS_DEF = [
    {
        "id": "single_snow_01",
        "user": (
            "폭설 때문에 도로가 많이 막힌 상황인데, "
            "서울 성북구/도봉구/강북구 쪽에서 지금 정상 운영 중인 응급실만 알려줘. "
            "임시 폐쇄되었거나 수용 불가인 곳은 빼줘."
        ),
        "meta": {
            "type": "Single-turn",
            "task": "폭설 상황에서 성북/도봉/강북구 정상 운영 응급실 필터링",
        },
    },
    {
        "id": "single_gangnam_01",
        "user": (
            "서울특별시 강남구 근처 응급실 실시간 가용 병상 상황이랑, "
            "중증질환 수용 가능한 곳도 같이 알려줘."
        ),
        "meta": {
            "type": "Single-turn",
            "task": "강남구 응급실 가용 병상 + 중증질환 수용 가능 병원 조회",
        },
    },
]

N_AUG_PER_SCENARIO = 3      # 시나리오당 증강 샘플 개수
AUG_TARGET_IDS = {          # 어떤 시나리오에 증강을 적용할지
    "single_snow_01",
    "single_gangnam_01",
}


if __name__ == "__main__":
    debug_log("==== 싱글턴 응급실 시나리오 (원본 + 증강) 생성 시작 ====")

    rows: List[Dict[str, Any]] = []

    for spec in SCENARIOS_DEF:
        base_id = spec["id"]
        base_user = spec["user"]
        base_meta = copy.deepcopy(spec["meta"])

        # 1) 원본 시나리오
        row = generate_single_scenario(
            scenario_id=base_id,
            user_content=base_user,
            scenario_meta=base_meta,
        )
        if row:
            rows.append(row)

        # 2) 증강 시나리오들
        if base_id in AUG_TARGET_IDS:
            for i in range(1, N_AUG_PER_SCENARIO + 1):
                aug_user = augment_text(base_user)
                aug_id = f"{base_id}_aug{i}"
                aug_meta = copy.deepcopy(spec["meta"])
                aug_meta.update(
                    {
                        "is_augmented": True,
                        "aug_index": i,
                        "augmented_from": base_id,
                    }
                )

                debug_log(f"[{base_id}] 증강 {i} 프롬프트: {aug_user}")

                aug_row = generate_single_scenario(
                    scenario_id=aug_id,
                    user_content=aug_user,
                    scenario_meta=aug_meta,
                )
                if aug_row:
                    rows.append(aug_row)

    if rows:
        out_path = str(OUTPUT_DIR / "single_emergency_scenarios_with_aug.jsonl")
        save_jsonl(rows, out_path)
        debug_log("완료 ✅")
    else:
        debug_log("어떤 시나리오도 생성되지 않음", level="ERROR")


[07:20:59] [INFO] ==== 싱글턴 응급실 시나리오 (원본 + 증강) 생성 시작 ====
[07:20:59] [INFO] 🚀 Solar 1차 호출 (single_snow_01)
[07:21:01] [INFO] ✅ tool_calls 개수: 3
[07:21:01] [INFO] 🛠️ execute_tool_call: get_emergency_room_list | args={'STAGE1': '서울특별시', 'STAGE2': '성북구'}
[07:21:01] [INFO] 🛠️ execute_tool_call: get_emergency_room_list | args={'STAGE1': '서울특별시', 'STAGE2': '도봉구'}
[07:21:01] [INFO] 🛠️ execute_tool_call: get_emergency_room_list | args={'STAGE1': '서울특별시', 'STAGE2': '강북구'}
[07:21:02] [INFO] 🚀 Solar 2차 호출 (single_snow_01) - 최종 답변

[single_snow_01] 최종 답변:
서울 성북구/도봉구/강북구 중 현재 정상 운영 중인 응급실은 다음과 같습니다. 임시 폐쇄 또는 수용 불가 병원은 제외하였습니다.

### 1. 성북구
- **병원명**: 학교법인고려중앙학원 고려대학교의과대학부속병원(안암병원)  
  - **응급실 연락처**: 02-920-5374  
  - **HPID**: A1100008  
  - **운영 상태**:  
    - 성인·중증응급실: 운영 중 (`hv5=Y`, `hv7=Y`)  
    - 가용 병상: 일반 4병상, 중환자실 13병상  
    - 특수 치료 가능:  
      ▶ 심근경색(협심증) `hvcrrtayn=Y`  
      ▶ 뇌졸중 `hvctayn=Y`  
      ▶ 외상 `hvangioayn=Y`  
      ▶ 중증화상 `hvventiayn=Y`  
    - **주의**: 산소공급기 `hvoxyayn=N1` (일부 제

# 2. Scenario Design (Multi-Trun)

## 2-1. Scenario Design 1 & 2 (Multi-Trun)

In [14]:
import os, sys, json, time, uuid, requests, xml.etree.ElementTree as ET, csv, random, copy
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any, Optional
from openai import OpenAI

# ======================================================================================
# 1. 공용 설정 + evaluate
# ======================================================================================
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
DEBUG_LOG_FILE = OUTPUT_DIR / "debug_log.txt"

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

try:
    from evaluate import evaluate_file
    print("✅ evaluate.py 모듈 로드 성공")
except ImportError:
    print("⚠️ evaluate.py 없음: 평가 기능이 건너뛰어집니다.")
    def evaluate_file(input_path, output_path): pass

# 2. API 키 설정 (환경 변수 사용 필수)
SOLAR_API_KEY = os.getenv("SOLAR_API_KEY", "")
SERVICE_KEY = os.getenv("SERVICE_KEY", "")

if SOLAR_API_KEY.startswith("up_"):
    client = OpenAI(api_key=SOLAR_API_KEY, base_url="https://api.upstage.ai/v1")
else:
    client = None
    print("⚠️ Solar API Key 미설정")

def debug_log(message: str, level: str = "INFO"):
    msg = f"[{datetime.now().strftime('%H:%M:%S')}] [{level}] {message}"
    print(msg)
    with open(DEBUG_LOG_FILE, "a", encoding="utf-8") as f:
        f.write(msg + "\n")

# ======================================================================================
# 2.5 프롬프트 증강 유틸 (여기가 새로 추가된 부분)
# ======================================================================================
random.seed(42)

SYNONYM_MAP = {
    "폭설": ["폭설 상황", "눈이 많이 오는 상황", "대설"],
    "도로": ["도로 상황", "길", "도로 상태"],
    "성북구": ["성북구 일대", "서울 성북구"],
    "도봉구": ["도봉구 일대", "서울 도봉구"],
    "강북구": ["강북구 일대", "서울 강북구"],
    "강남구": ["강남구 일대", "서울 강남구"],
    "응급실": ["응급센터", "응급 진료실", "응급실(ER)"],
    "권역외상센터": ["권역 외상 센터", "권역응급외상센터"],
    "외상센터": ["외상 센터", "외상 전문 센터"],
    "정상 운영": ["현재 정상 운영 중인", "지금 운영 중인", "지금 열려 있는"],
    "수용 불가": ["수용이 어려운", "진료가 어려운", "받기 힘든"],
    "중증질환": ["중증 질환", "심각한 질환"],
}

def random_deletion(words, p=0.08):
    if len(words) == 1:
        return words
    new_words = [w for w in words if random.uniform(0, 1) > p]
    if not new_words:
        new_words.append(random.choice(words))
    return new_words

def random_swap(words, n_swaps=1):
    if len(words) < 2:
        return words
    new_words = words[:]
    for _ in range(n_swaps):
        i, j = random.sample(range(len(new_words)), 2)
        new_words[i], new_words[j] = new_words[j], new_words[i]
    return new_words

def synonym_replacement(text: str, replace_prob: float = 0.4) -> str:
    new_text = text
    for key, syn_list in SYNONYM_MAP.items():
        if key in new_text and random.random() < replace_prob:
            synonym = random.choice(syn_list)
            new_text = new_text.replace(key, synonym)
    return new_text

def augment_text(
    text: str,
    alpha_del: float = 0.08,
    max_swaps: int = 2,
    synonym_prob: float = 0.4,
) -> str:
    # 1) 유의어 치환
    text = synonym_replacement(text, replace_prob=synonym_prob)

    # 2) 단어 삭제 / 스왑
    words = text.split()
    if len(words) <= 3:
        return text

    words = random_deletion(words, p=alpha_del)
    n_swaps = min(max_swaps, max(1, len(words) // 12))
    words = random_swap(words, n_swaps=n_swaps)
    return " ".join(words)

# ======================================================================================
# 3. 도구(Tools) 정의
# ======================================================================================
API_URLS = {
    "emergency_list": "https://apis.data.go.kr/B552657/ErmctInfoInqireService/getEmrrmRltmUsefulSckbdInfoInqire",
    "trauma_center": "https://apis.data.go.kr/B552657/ErmctInfoInqireService/getTrtmListInfoInqire",
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_emergency_room_list",
            "description": "지역(시도, 시군구)을 입력받아 실시간 응급실 가용 병상 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "STAGE1": {"type": "string", "description": "시/도 (예: 서울특별시)"},
                    "STAGE2": {"type": "string", "description": "시/군/구 (예: 강남구)"}
                },
                "required": ["STAGE1", "STAGE2"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_hospital_basic_info_by_hpid",
            "description": "병원 ID(HPID)를 입력받아 뇌출혈, 뇌경색 등 중증질환 수용 가능 여부를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {"HPID": {"type": "string", "description": "병원 식별 코드 (예: A1100008)"}},
                "required": ["HPID"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_trauma_center_list",
            "description": "지역 정보를 입력받아 중증 외상 환자 전용 권역외상센터 목록을 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "Q0": {"type": "string", "description": "시/도"},
                    "Q1": {"type": "string", "description": "시/군/구"}
                },
                "required": ["Q0", "Q1"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_er_message_by_hpid",
            "description": "병원 ID(HPID)로 응급실의 실시간 공지사항(수용 불가 메시지 등)을 확인합니다.",
            "parameters": {
                "type": "object",
                "properties": {"HPID": {"type": "string", "description": "병원 식별 코드"}},
                "required": ["HPID"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_travel_time",
            "description": "출발지에서 병원까지의 예상 이동 시간을 계산합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "출발지"},
                    "destination": {"type": "string", "description": "도착 병원명"}
                },
                "required": ["origin", "destination"]
            }
        }
    }
]

# ======================================================================================
# 4. 도구 실행기
# ======================================================================================
def execute_tool_call(name, args):
    debug_log(f"🛠️ 실행: {name} | 파라미터: {args}")
    
    # 1. 가상 도구
    if name == "get_travel_time":
        return json.dumps({
            "status": "OK", 
            "time_minutes": 15 + len(args.get("destination", ""))}
        , ensure_ascii=False)

    # 2. HPID 기반 조회 (Mock Data)
    if "by_hpid" in name:
        hpid = args.get("HPID", "")
        mock_db = {
            "A1100008": {"name": "삼성서울병원", "stroke": "Y", "msg": "소아 응급 불가"},
            "A1100010": {"name": "강남세브란스", "stroke": "Y", "msg": "정상 운영"}
        }
        info = mock_db.get(hpid, {"name": "알수없음", "stroke": "N", "msg": "정보 없음"})
        return json.dumps({"status": "OK", "result": info}, ensure_ascii=False)

    # 3. 실제 API 호출 (키 없으면 Mock)
    if "YOUR_" in SERVICE_KEY:
        return json.dumps({
            "status": "OK",
            "items": [
                {"dutyName": "삼성서울병원", "hpid": "A1100008", "hvec": "5"},
                {"dutyName": "강남세브란스", "hpid": "A1100010", "hvec": "3"}
            ]
        }, ensure_ascii=False)

    # 실제 호출
    url = API_URLS.get("emergency_list" if "emergency" in name else "trauma_center")
    params = {"serviceKey": SERVICE_KEY, "numOfRows": "10"}
    if "STAGE1" in args: params["STAGE1"] = args["STAGE1"]
    if "STAGE2" in args: params["STAGE2"] = args["STAGE2"]
    if "Q0" in args: params["Q0"] = args["Q0"]
    if "Q1" in args: params["Q1"] = args["Q1"]

    try:
        res = requests.get(url, params=params, timeout=5)
        return json.dumps({"status": "OK", "raw": res.text[:200]}, ensure_ascii=False)
    except Exception as e:
        return json.dumps({"status": "ERROR", "message": str(e)}, ensure_ascii=False)

# ======================================================================================
# 5. 시나리오 생성기 (evaluate.py 스키마 완벽 대응)
# ======================================================================================
def generate_scenario(s_id, turns, meta):
    debug_log(f"🚀 시나리오 생성: {s_id}")
    
    messages = [{
        "role": "system", 
        "content": "당신은 응급의료 AI입니다. 사용자 질문에 답하기 위해 반드시 도구를 사용해야 합니다."
    }]
    
    conversation = []  # evaluate.py는 이걸 'messages'로 인식해야 함

    for i, user_text in enumerate(turns):
        messages.append({"role": "user", "content": user_text})
        conversation.append({"role": "user", "content": user_text})
        
        try:
            resp = client.chat.completions.create(
                model="solar-pro2",
                messages=messages,
                tools=TOOLS,
                tool_choice="required"  # 강제 호출
            )
            ai_msg = resp.choices[0].message
        except Exception as e:
            debug_log(f"API Error: {e}", "ERROR")
            return None

        if ai_msg.tool_calls:
            debug_log(f"✅ Tool Call 발생: {len(ai_msg.tool_calls)}건")
            
            tool_calls_list = []
            for tc in ai_msg.tool_calls:
                tool_calls_list.append({
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                })
            
            conversation.append({
                "role": "assistant",
                "content": ai_msg.content or "",
                "tool_calls": tool_calls_list
            })
            messages.append(ai_msg)

            for tc in ai_msg.tool_calls:
                args = json.loads(tc.function.arguments)
                result = execute_tool_call(tc.function.name, args)
                
                tool_msg = {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result
                }
                conversation.append(tool_msg)
                messages.append(tool_msg)

            final_resp = client.chat.completions.create(
                model="solar-pro2",
                messages=messages,
                tools=TOOLS
            )
            final_msg = final_resp.choices[0].message
            conversation.append({
                "role": "assistant",
                "content": final_msg.content
            })
            messages.append(final_msg)
        else:
            debug_log("❌ 경고: 도구가 호출되지 않음", "WARN")
    
    return {
        "id": s_id,
        "row": {
            "messages": conversation,
            "metadata": json.dumps({"tools": TOOLS, **meta}, ensure_ascii=False)
        }
    }

# ======================================================================================
# 6. 실행 및 저장 (+ 프롬프트 증강 포함)
# ======================================================================================
SCENARIOS = [
    {
        "id": "single_01",
        "turns": ["서울 강남구 응급실 목록과 삼성서울병원(HPID: A1100008) 뇌졸중 수술 가능 여부 확인해줘."],
        "meta": {"type": "Single-turn", "task": "응급실 조회"}
    },
    {
        "id": "multi_01",
        "turns": [
            "인천 남동구 권역외상센터 찾아줘.",
            "거기까지 인천터미널에서 얼마나 걸려?"
        ],
        "meta": {"type": "Multi-turn", "task": "외상센터 및 이동시간"}
    }
]

N_AUG_PER_SCENARIO = 3  # 각 시나리오당 증강 몇 개 만들지

results = []
if client:
    for s in SCENARIOS:
        base_id = s["id"]
        base_turns = s["turns"]
        base_meta = s["meta"]

        # 1) 원본 시나리오
        res = generate_scenario(base_id, base_turns, base_meta)
        if res:
            results.append(res)

        # 2) 증강 시나리오들
        for i in range(1, N_AUG_PER_SCENARIO + 1):
            aug_turns = [augment_text(t) for t in base_turns]
            aug_meta = dict(base_meta)
            aug_meta.update({
                "is_augmented": True,
                "aug_index": i,
                "augmented_from": base_id,
            })
            aug_id = f"{base_id}_aug{i}"

            debug_log(f"[{base_id}] 증강 {i}턴: {aug_turns}")

            res_aug = generate_scenario(aug_id, aug_turns, aug_meta)
            if res_aug:
                results.append(res_aug)

    # 저장
    out_path = OUTPUT_DIR / "dataset_with_aug.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    
    print(f"\n✅ 데이터셋 생성 완료 (원본+증강): {len(results)}개")
    
    # 평가
    print("📊 평가 시작...")
    eval_path = OUTPUT_DIR / "eval_result_with_aug.json"
    evaluate_file(str(out_path), str(eval_path))
    print(f"🎉 평가 완료! 결과 파일: {eval_path}")
    
    try:
        with open(eval_path, "r", encoding="utf-8") as f:
            res = json.load(f)
            print(json.dumps(res[:2], indent=2, ensure_ascii=False))
    except:
        pass

else:
    print("🚨 API Key가 없어 실행할 수 없습니다.")


⚠️ evaluate.py 없음: 평가 기능이 건너뛰어집니다.
[07:21:43] [INFO] 🚀 시나리오 생성: single_01
[07:21:45] [INFO] ✅ Tool Call 발생: 2건
[07:21:45] [INFO] 🛠️ 실행: get_emergency_room_list | 파라미터: {'STAGE1': '서울특별시', 'STAGE2': '강남구'}
[07:21:45] [INFO] 🛠️ 실행: get_hospital_basic_info_by_hpid | 파라미터: {'HPID': 'A1100008'}
[07:21:47] [INFO] [single_01] 증강 1턴: ['서울 강남구 응급 진료실 목록과 삼성서울병원(HPID: 가능 수술 뇌졸중 여부 확인해줘.']
[07:21:47] [INFO] 🚀 시나리오 생성: single_01_aug1
[07:21:49] [INFO] ✅ Tool Call 발생: 2건
[07:21:49] [INFO] 🛠️ 실행: get_emergency_room_list | 파라미터: {'STAGE1': '서울특별시', 'STAGE2': '강남구'}
[07:21:49] [INFO] 🛠️ 실행: get_hospital_basic_info_by_hpid | 파라미터: {'HPID': 'A1100008'}
[07:21:51] [INFO] [single_01] 증강 2턴: ['서울 강남구 응급실 목록과 삼성서울병원(HPID: A1100008) 뇌졸중 가능 수술 여부 확인해줘.']
[07:21:51] [INFO] 🚀 시나리오 생성: single_01_aug2
[07:21:52] [INFO] ✅ Tool Call 발생: 2건
[07:21:52] [INFO] 🛠️ 실행: get_emergency_room_list | 파라미터: {'STAGE1': '서울특별시', 'STAGE2': '강남구'}
[07:21:52] [INFO] 🛠️ 실행: get_hospital_basic_info_by_hpid | 파라미터: {'HPID': 'A1100008'

# 3. MASKING

In [15]:
# ======================================================================================
# 🏥 폭설 상황 싱글턴 시나리오
#    (Solar-Pro2 + 공공데이터 API + BFCL 포맷 저장
#     + 프롬프트 증강 + 입력/출력 마스킹)
# ======================================================================================
import os
import json
import random
import requests
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Optional
from openai import OpenAI

# --------------------------------------------------
# 0. 공용 유틸
# --------------------------------------------------
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
DEBUG_LOG_FILE = OUTPUT_DIR / "debug_log_snow_single.txt"


def debug_log(msg: str, level: str = "INFO") -> None:
    line = f"[{datetime.now().strftime('%H:%M:%S')}] [{level}] {msg}"
    print(line)
    with open(DEBUG_LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def save_jsonl(items: List[Dict[str, Any]], path: str):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")
    print(f"✅ saved: {path}")


# ======================================================================================
# 🔹 A. 프롬프트 증강 유틸 (원본 → 여러 개 변형 프롬프트)
# ======================================================================================

random.seed(42)  # 재현성 확보

SYNONYM_MAP = {
    # 상황 / 날씨
    "폭설": ["폭설 상황", "눈이 많이 오는 상황", "대설"],
    "도로": ["도로 상황", "길", "도로 상태"],
    # 지역
    "성북구": ["성북구 일대", "서울 성북구"],
    "도봉구": ["도봉구 일대", "서울 도봉구"],
    "강북구": ["강북구 일대", "서울 강북구"],
    # 의료/응급
    "응급실": ["응급센터", "응급 진료실", "응급실(ER)"],
    "병원": ["의료기관", "병원 기관"],
    "병상": ["입원 병상", "병상 수"],
    "수용 불가": ["수용이 어려운", "진료가 어려운", "받기 힘든"],
    "정상 운영": ["정상으로 운영 중인", "현재 운영 중인", "지금 열려 있는"],
}


def random_deletion(words, p=0.08):
    """단어들을 확률 p로 랜덤 삭제"""
    if len(words) == 1:
        return words
    new_words = [w for w in words if random.uniform(0, 1) > p]
    if not new_words:
        new_words.append(random.choice(words))
    return new_words


def random_swap(words, n_swaps=1):
    """문장 내 단어 위치를 n_swaps번 랜덤으로 교환"""
    if len(words) < 2:
        return words
    new_words = words[:]
    for _ in range(n_swaps):
        i, j = random.sample(range(len(new_words)), 2)
        new_words[i], new_words[j] = new_words[j], new_words[i]
    return new_words


def synonym_replacement(text: str, replace_prob: float = 0.4) -> str:
    """
    SYNONYM_MAP 기반 간단 유의어 치환
    """
    new_text = text
    for key, syn_list in SYNONYM_MAP.items():
        if key in new_text and random.random() < replace_prob:
            synonym = random.choice(syn_list)
            new_text = new_text.replace(key, synonym)
    return new_text


def augment_text(
    text: str,
    alpha_del: float = 0.08,
    max_swaps: int = 2,
    synonym_prob: float = 0.4,
) -> str:
    """
    1) 유의어 치환 → 2) 약간의 단어 삭제 → 3) 약간의 단어 위치 스왑
    """
    # 1) 유의어 치환
    text = synonym_replacement(text, replace_prob=synonym_prob)

    words = text.split()
    if len(words) <= 3:
        return text

    # 2) 삭제
    words = random_deletion(words, p=alpha_del)

    # 3) 스왑
    n_swaps = min(max_swaps, max(1, len(words) // 12))
    words = random_swap(words, n_swaps=n_swaps)

    return " ".join(words)


# ======================================================================================
# 🔹 B. 마스킹 유틸 (LLM 호출 전/후 한 번씩 적용)
#     - 쿼리/답변 안의 특정 토큰을 토큰화된 마스크로 치환
# ======================================================================================

KEYWORD_MASK_MAP: Dict[str, str] = {
    # 지역(조합 순서 중요: 긴 문자열 → 짧은 문자열 순으로)
    "서울 성북구": "[LOC_성북구]",
    "서울 도봉구": "[LOC_도봉구]",
    "서울 강북구": "[LOC_강북구]",
    "성북구": "[LOC_성북구]",
    "도봉구": "[LOC_도봉구]",
    "강북구": "[LOC_강북구]",
    "서울": "[CITY_서울]",

    # (필요하면 병원명/HPID도 추가)
    "삼성서울병원": "[HOSP_1]",
    "강남세브란스": "[HOSP_2]",
    "A1100008": "[HPID_1]",
    "A1100010": "[HPID_2]",
}


def mask_text(text: str) -> str:
    """
    KEYWORD_MASK_MAP에 있는 토큰을 마스크 토큰으로 일괄 치환.
    - 랜덤 삭제/스왑 같은 건 절대 안 함 (순수 마스킹 용도)
    - end-to-end에서 한 번만 돌리는 PII 마스킹 모델 대체용
    """
    masked = text
    # 긴 키부터 치환되도록 정렬 (부분 문자열 겹침 방지)
    for key in sorted(KEYWORD_MASK_MAP.keys(), key=len, reverse=True):
        masked = masked.replace(key, KEYWORD_MASK_MAP[key])
    return masked


# ======================================================================================
# 3. Tools (BFCL 호환 스키마)
#    (여기 위/아래로 SOLAR_API_KEY, SERVICE_KEY, API_URLS, client 정의는
#     네가 기존에 쓰던 그대로 두면 됨)
# ======================================================================================
TOOLS: List[Dict[str, Any]] = [
    {
        "type": "function",
        "function": {
            "name": "get_emergency_room_list",
            "description": "시/도·시군구 단위로 실시간 응급실 가용 병상 및 운영 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "STAGE1": {
                        "type": "string",
                        "description": "시/도명 (예: 서울특별시)",
                    },
                    "STAGE2": {
                        "type": "string",
                        "description": "시/군/구명 (예: 성북구, 도봉구, 강북구)",
                    },
                },
                "required": ["STAGE1", "STAGE2"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_er_message_by_hpid",
            "description": "HPID로 해당 병원의 응급실 실시간 메시지(임시 폐쇄, 수용 불가 등)를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "HPID": {
                        "type": "string",
                        "description": "병원 ID (예: A1100008)",
                    }
                },
                "required": ["HPID"],
            },
        },
    },
]

# --------------------------------------------------
# 4. 도구 실행 함수: Solar가 호출한 tool을 실제 API로 연결
#    (SERVICE_KEY, API_URLS, client 등은 네가 기존에 선언한 그대로 사용)
# --------------------------------------------------
def execute_tool_call(name: str, args: Dict[str, Any]) -> str:
    debug_log(f"🛠️ execute_tool_call: {name} | args={args}")

    try:
        if name == "get_emergency_room_list":
            params = {
                "serviceKey": SERVICE_KEY,
                "STAGE1": args["STAGE1"],
                "STAGE2": args["STAGE2"],
                "pageNo": "1",
                "numOfRows": "100",
            }
            res = requests.get(API_URLS["emergency_list"], params=params, timeout=5)
            res.raise_for_status()
            return json.dumps(
                {
                    "status": "OK",
                    "api": "getEmrrmRltmUsefulSckbdInfoInqire",
                    "stage1": args["STAGE1"],
                    "stage2": args["STAGE2"],
                    "raw_xml": res.text,
                },
                ensure_ascii=False,
            )

        elif name == "get_er_message_by_hpid":
            params = {
                "serviceKey": SERVICE_KEY,
                "HPID": args["HPID"],
                "pageNo": "1",
                "numOfRows": "20",
            }
            res = requests.get(API_URLS["er_message"], params=params, timeout=5)
            res.raise_for_status()
            return json.dumps(
                {
                    "status": "OK",
                    "api": "getEmrrmSrsillDissMsgInqire",
                    "hpid": args["HPID"],
                    "raw_xml": res.text,
                },
                ensure_ascii=False,
            )

        else:
            return json.dumps(
                {"status": "ERROR", "message": f"Unknown tool: {name}"},
                ensure_ascii=False,
            )

    except Exception as e:
        debug_log(f"❌ tool 실행 실패: {e}", level="ERROR")
        return json.dumps(
            {"status": "ERROR", "message": f"{type(e).__name__}: {e}"},
            ensure_ascii=False,
        )


# --------------------------------------------------
# 5. 싱글턴 시나리오 생성 (프롬프트를 인자로 받아서 사용)
#     ⚠️ 이 시점의 user_text는 이미 "증강 + 마스킹"까지 끝난 상태라고 가정
# --------------------------------------------------
def generate_snow_single_scenario(
    user_text: str,
    scenario_id: str,
    is_augmented: bool = False,
    aug_index: int = 0,
    base_scenario_id: Optional[str] = None,
) -> Optional[Dict[str, Any]]:
    """
    user_text: 실제 Solar에 던질 유저 프롬프트
              (원본/증강 둘 다 가능, 이미 마스킹까지 된 상태로 받음)
    scenario_id: 이 row에 부여할 id (예: single_snow_01, single_snow_01_aug1)
    """

    # BFCL-style 메시지 로그
    conversation: List[Dict[str, Any]] = []

    # 1) system + user
    system_msg = {
        "role": "system",
        "content": (
            "당신은 한국의 응급의료 상황을 도와주는 AI 비서입니다. "
            "사용자의 질문에 답변할 때는 반드시 제공된 도구를 사용해 "
            "실시간 정보를 조회한 후, 그 결과를 바탕으로 답변해야 합니다."
        ),
    }
    user_msg = {"role": "user", "content": user_text}  # 이미 마스킹된 쿼리

    conversation.append(system_msg)
    conversation.append(user_msg)

    messages_for_solar = [system_msg, user_msg]

    # 2) 첫 번째 호출: 반드시 tool_call 생성
    try:
        debug_log(f"🚀 Solar 1차 호출 (tool_choice=required) | {scenario_id}")
        first = client.chat.completions.create(
            model="solar-pro2",
            messages=messages_for_solar,
            tools=TOOLS,
            tool_choice="required",
        )
    except Exception as e:
        debug_log(f"Solar 1차 호출 실패: {e}", level="ERROR")
        return None

    ai_msg = first.choices[0].message

    tool_calls_list: List[Dict[str, Any]] = []
    if ai_msg.tool_calls:
        debug_log(f"✅ tool_calls 개수: {len(ai_msg.tool_calls)}")
        for tc in ai_msg.tool_calls:
            tool_calls_list.append(
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
            )
    else:
        debug_log("⚠️ tool_calls 없음 (원래는 나오도록 설계돼야 함)", level="WARN")

    assistant_with_tools_msg = {
        "role": "assistant",
        "content": ai_msg.content or "",
        "tool_calls": tool_calls_list if tool_calls_list else [],
    }
    conversation.append(assistant_with_tools_msg)
    messages_for_solar.append(ai_msg)

    # 3) tool_calls 실제 실행 후, tool 메시지들 추가
    if ai_msg.tool_calls:
        for tc in ai_msg.tool_calls:
            try:
                args = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                debug_log(
                    f"❌ arguments JSON 파싱 실패: {tc.function.arguments}",
                    level="ERROR",
                )
                continue

            tool_result = execute_tool_call(tc.function.name, args)

            tool_msg = {
                "role": "tool",
                "tool_call_id": tc.id,
                "content": tool_result,
            }
            conversation.append(tool_msg)
            messages_for_solar.append(tool_msg)

    # 4) 두 번째 호출: 최종 자연어 답변
    try:
        debug_log(f"🚀 Solar 2차 호출 (최종 답변) | {scenario_id}")
        second = client.chat.completions.create(
            model="solar-pro2",
            messages=messages_for_solar,
            tools=TOOLS,
        )
    except Exception as e:
        debug_log(f"Solar 2차 호출 실패: {e}", level="ERROR")
        return None

    final_ai_msg = second.choices[0].message
    raw_answer = final_ai_msg.content or ""
    masked_answer = mask_text(raw_answer)  # 🔴 생성 후 마스킹 한 번

    final_assistant_msg = {
        "role": "assistant",
        "content": masked_answer,
    }
    conversation.append(final_assistant_msg)

    # 5) metadata 구성 (증강/마스킹 여부 표시)
    if base_scenario_id is None:
        base_scenario_id = scenario_id

    metadata_obj = {
        "tools": TOOLS,
        "scenario_meta": {
            "type": "Single-turn",
            "task": "폭설 상황에서 성북/도봉/강북구 정상 운영 응급실 필터링",
            "augmentation": {
                "is_augmented": is_augmented,
                "aug_index": aug_index if is_augmented else 0,
                "base_scenario_id": base_scenario_id,
            },
            "masking": {
                "input_masked": True,   # 쿼리 마스킹 O
                "output_masked": True,  # 답변 마스킹 O
            },
        },
    }

    row = {
        "id": scenario_id,
        "row": {
            "messages": conversation,
            "metadata": json.dumps(metadata_obj, ensure_ascii=False),
        },
    }

    return row


# --------------------------------------------------
# 6. 메인: 쿼리 → 증강 → (마스킹) → Solar → (마스킹) → 답변
# --------------------------------------------------
if __name__ == "__main__":
    debug_log("==== 폭설 싱글턴 시나리오 (원본 + 증강, 마스킹 포함) 생성 시작 ====")

    base_scenario_id = "single_snow_01"
    # 🔴 시나리오 쿼리는 그대로 유지 (여기 건드리지 않음)
    base_user_text = (
        "폭설 때문에 도로가 많이 막힌 상황인데, "
        "서울 성북구/도봉구/강북구 쪽에서 지금 정상 운영 중인 응급실만 알려줘. "
        "임시 폐쇄되었거나 수용 불가인 곳은 빼줘."
    )

    N_AUG = 3  # 원본 1개 + 증강 3개 → 총 4개 시나리오

    rows: List[Dict[str, Any]] = []

    # 1) 원본 프롬프트 시나리오
    #    base_user_text -> (증강 없이 그대로) -> 마스킹 -> LLM
    base_masked_text = mask_text(base_user_text)
    base_row = generate_snow_single_scenario(
        user_text=base_masked_text,
        scenario_id=base_scenario_id,
        is_augmented=False,
    )
    if base_row:
        rows.append(base_row)

    # 2) 증강 프롬프트 시나리오들
    for i in range(1, N_AUG + 1):
        # 쿼리 -> 증강
        aug_text = augment_text(base_user_text)
        # 증강된 쿼리 -> 마스킹
        aug_masked = mask_text(aug_text)
        aug_id = f"{base_scenario_id}_aug{i}"

        debug_log(f"--- 증강 프롬프트 {i} (원본): {aug_text}")
        debug_log(f"--- 증강 프롬프트 {i} (마스킹): {aug_masked}")

        aug_row = generate_snow_single_scenario(
            user_text=aug_masked,
            scenario_id=aug_id,
            is_augmented=True,
            aug_index=i,
            base_scenario_id=base_scenario_id,
        )
        if aug_row:
            rows.append(aug_row)

    if not rows:
        debug_log("시나리오 생성 실패 (row 없음)", level="ERROR")
    else:
        out_path = str(OUTPUT_DIR / "snow_single_turn_dataset_with_aug_and_mask.jsonl")
        save_jsonl(rows, out_path)
        debug_log("완료 ✅")


[07:22:09] [INFO] ==== 폭설 싱글턴 시나리오 (원본 + 증강, 마스킹 포함) 생성 시작 ====
[07:22:09] [INFO] 🚀 Solar 1차 호출 (tool_choice=required) | single_snow_01
[07:22:11] [INFO] ✅ tool_calls 개수: 3
[07:22:11] [INFO] 🛠️ execute_tool_call: get_emergency_room_list | args={'STAGE1': '서울특별시', 'STAGE2': '성북구'}
[07:22:11] [INFO] 🛠️ execute_tool_call: get_emergency_room_list | args={'STAGE1': '서울특별시', 'STAGE2': '도봉구'}
[07:22:11] [INFO] 🛠️ execute_tool_call: get_emergency_room_list | args={'STAGE1': '서울특별시', 'STAGE2': '강북구'}
[07:22:11] [INFO] 🚀 Solar 2차 호출 (최종 답변) | single_snow_01
[07:22:15] [INFO] --- 증강 프롬프트 1 (원본): 폭설 때문에 중인 막힌 상황인데, 서울 성북구 일대/도봉구/강북구 지금 정상으로 운영 중인 많이 응급실만 알려줘. 임시 폐쇄되었거나 진료가 어려운인 곳은 빼줘.
[07:22:15] [INFO] --- 증강 프롬프트 1 (마스킹): 폭설 때문에 중인 막힌 상황인데, [LOC_[LOC_성북구]] 일대/[LOC_도봉구]/[LOC_강북구] 지금 정상으로 운영 중인 많이 응급실만 알려줘. 임시 폐쇄되었거나 진료가 어려운인 곳은 빼줘.
[07:22:15] [INFO] 🚀 Solar 1차 호출 (tool_choice=required) | single_snow_01_aug1
[07:22:16] [INFO] ✅ tool_calls 개수: 3
[07:22:16] [INFO] 🛠️ execute_tool_call: get_emergency_

# 4. BCLF

In [ ]:
# ============================================================
# 최종 데이터셋 평가 & 저장
# ============================================================

print("\n" + "=" * 60)
print("📊 전체 평가 결과 통합")
print("=" * 60)

# 모든 평가 결과 통합
all_results = single_turn_results + multi_turn_results
all_evaluation_results = []

# 통과한 시나리오만 원본 데이터와 매칭
passed_scenarios = []

# Single turn 통과 항목
for result in single_turn_results:
    if result.get("pass"):
        # 원본 데이터 찾기
        entry_id = result.get("id")
        original_entry = next((e for e in single_turn_data if (e.get("row_idx") or e.get("id")) == entry_id), None)
        if original_entry:
            original_entry["evaluation"] = result
            passed_scenarios.append(original_entry)

# Multi turn 통과 항목
for result in multi_turn_results:
    if result.get("pass"):
        # 원본 데이터 찾기
        entry_id = result.get("id")
        original_entry = next((e for e in multi_turn_data if (e.get("row_idx") or e.get("id")) == entry_id), None)
        if original_entry:
            original_entry["evaluation"] = result
            passed_scenarios.append(original_entry)

# 평가 결과 포맷팅 (CSV용)
for result in all_results:
    all_evaluation_results.append({
        "id": result.get("id"),
        "correct_function_name": result.get("correct_function_name", False),
        "valid_arguments": result.get("valid_arguments", False),
        "no_hallucinated_calls": result.get("no_hallucinated_calls", False),
        "pass": result.get("pass", False),
        "num_turns": result.get("num_turns", 0),
        "total_tool_calls": result.get("total_tool_calls", 0)
    })

# 통계 계산
total = len(all_results)
passed = sum(1 for r in all_results if r.get("pass"))
failed = total - passed

single_turn_passed = sum(1 for r in single_turn_results if r.get("pass"))
multi_turn_passed = sum(1 for r in multi_turn_results if r.get("pass"))

print(f"전체 시나리오: {total}개")
print(f"  - Single Turn: {len(single_turn_results)}개 (통과: {single_turn_passed}개)")
print(f"  - Multi Turn: {len(multi_turn_results)}개 (통과: {multi_turn_passed}개)")
print(f"\n✅ 총 통과: {passed}개 ({passed/total*100:.1f}%)" if total > 0 else "\n✅ 총 통과: 0개")
print(f"❌ 총 실패: {failed}개 ({failed/total*100:.1f}%)" if total > 0 else "❌ 총 실패: 0개")
print("=" * 60)

# --------------------------
# 최종 데이터셋 저장
# --------------------------
print("\n" + "=" * 60)
print("💾 최종 데이터셋 저장")
print("=" * 60)

# 1. JSONL 형식으로 통과 항목만 저장
output_jsonl = "outputs/final_dataset.jsonl"
with open(output_jsonl, "w", encoding="utf-8") as f:
    for scenario in passed_scenarios:
        f.write(json.dumps(scenario, ensure_ascii=False) + "\n")
print(f"✅ JSONL 저장: {output_jsonl} (통과 항목: {len(passed_scenarios)}개)")

# 2. JSON 형식으로 전체 저장
final_dataset = {
    "project": "Upstage AI Agent QA 시나리오 데이터셋 구축",
    "domain": SELECTED_DOMAIN,
    "created_at": datetime.now().isoformat(),
    "tools": tools,
    "scenarios": passed_scenarios,
    "statistics": {
        "total_generated": total,
        "passed_evaluation": len(passed_scenarios),
        "single_turn_total": len(single_turn_results),
        "single_turn_passed": single_turn_passed,
        "multi_turn_total": len(multi_turn_results),
        "multi_turn_passed": multi_turn_passed,
        "pass_rate": f"{passed/total*100:.1f}%" if total > 0 else "0%"
    }
}

output_json = "outputs/final_dataset.json"
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(final_dataset, f, ensure_ascii=False, indent=2)
print(f"✅ JSON 저장: {output_json}")

# 3. 평가 리포트 CSV 저장
report_csv = "artifacts/bfcl_evaluation_report.csv"
with open(report_csv, "w", newline="", encoding="utf-8") as f:
    fieldnames = [
        "id", "correct_function_name", "valid_arguments", 
        "no_hallucinated_calls", "pass", "num_turns", "total_tool_calls"
    ]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for result in all_evaluation_results:
        row = {
            "id": result.get("id"),
            "correct_function_name": 1 if result.get("correct_function_name") else 0,
            "valid_arguments": 1 if result.get("valid_arguments") else 0,
            "no_hallucinated_calls": 1 if result.get("no_hallucinated_calls") else 0,
            "pass": 1 if result.get("pass") else 0,
            "num_turns": result.get("num_turns", 0),
            "total_tool_calls": result.get("total_tool_calls", 0)
        }
        writer.writerow(row)
print(f"✅ CSV 리포트: {report_csv}")

print("\n" + "=" * 60)
print("🎉 모든 작업 완료!")
print("=" * 60)
print(f"\n📁 생성된 파일:")
print(f"   1. {output_jsonl} ({len(passed_scenarios)}개 통과 항목)")
print(f"   2. {output_json}")
print(f"   3. {report_csv}")
print(f"\n📊 최종 통계:")
print(f"   - Single-turn 통과: {single_turn_passed}/{len(single_turn_results)}개")
print(f"   - Multi-turn 통과: {multi_turn_passed}/{len(multi_turn_results)}개")
print(f"   - 전체 통과율: {passed/total*100:.1f}%" if total > 0 else "   - 전체 통과율: 0%")
print("\n✨정성평가를 진행보세요!")
